<p><font size="6" color='grey'> <b>
KI-Agenten. Verstehen. Anwenden. Gestalten.
</b></font> </br></p>



<p><font size="5" color='grey'> <b>
Production Deployment
</b></font> </br></p>

---

In [1]:
#@title 🔧 Umgebung einrichten{ display-mode: "form" }
!uv pip install --system -q git+https://github.com/ralf-42/Agenten.git#subdirectory=04_modul
!uv pip install --system -q fastapi uvicorn httpx

import os
os.environ["LANGSMITH_TRACING"]  = "true"
os.environ["LANGSMITH_PROJECT"]  = "M34-Production-Deployment"
os.environ["LANGSMITH_ENDPOINT"] = "https://eu.api.smith.langchain.com"

from genai_lib.utilities import (
    check_environment,
    get_ipinfo,
    setup_api_keys,
    mprint,
    install_packages,
    mermaid,
    get_model_profile,
    extract_thinking,
    load_prompt,
    show_trace
)

setup_api_keys(['OPENAI_API_KEY', 'LANGSMITH_API_KEY'], create_globals=False)
print()
check_environment()
print()
get_ipinfo()

✗ Fehler beim Setzen von OPENAI_API_KEY: Requesting secret OPENAI_API_KEY timed out. Secrets can only be fetched when running from the Colab UI.
✗ Fehler beim Setzen von LANGSMITH_API_KEY: Requesting secret LANGSMITH_API_KEY timed out. Secrets can only be fetched when running from the Colab UI.

Python Version: 3.12.13 (main, Mar  4 2026, 09:23:07) [GCC 11.4.0]

Installierte LangChain- und LangGraph-Bibliotheken:
langchain                                1.2.12
langchain-chroma                         1.1.0
langchain-classic                        1.0.3
langchain-community                      0.4.1
langchain-core                           1.2.19
langchain-ollama                         1.0.1
langchain-openai                         1.1.11
langchain-text-splitters                 1.1.1
langgraph                                1.1.2
langgraph-checkpoint                     4.0.1
langgraph-prebuilt                       1.0.8
langgraph-sdk                            0.3.11

IP-Adresse: 34

# 1 | Übersicht
---

<p><font color='black' size="5">
Vom Notebook zur Production
</font></p>

Ein Notebook-Agent und ein Production-Agent lösen dieselbe Aufgabe — aber mit
grundlegend verschiedenen Anforderungen:

| Dimension | Notebook (Entwicklung) | Production |
|-----------|----------------------|------------|
| **API-Keys** | Hardcoded / setup_api_keys() | Umgebungsvariablen (`.env`, Secrets) |
| **Fehler** | Exception → Notebook stoppt | Caught → strukturiertes Logging |
| **Skalierung** | 1 User, manuell | N User, automatisch |
| **Deployment** | Jupyter starten | Docker Container / Cloud |
| **Schnittstelle** | Zelle ausführen | REST API (`/invoke`, `/stream`) |
| **Monitoring** | `print()` | LangSmith + Structured Logging |
| **Kosten** | Ignoriert | Gemessen und begrenzt |


# 2 | Von Notebook zu Production
---


<p><font color='black' size="5">
Production-Checkliste
</font></p>

Bevor ein Agent deployed wird, müssen diese fünf Punkte erfüllt sein:

```
✅ 1. API-Keys aus Umgebungsvariablen (nie hardcoded)
✅ 2. Strukturiertes Logging (kein print())
✅ 3. Fehlerbehandlung mit Fallback-Antworten
✅ 4. Timeout und Retry-Logik
✅ 5. Observability: LangSmith Tracing aktiv
```




<p><font color='black' size="5">
Pattern: Production Agent Wrapper
</font></p>

```python
# ❌ Notebook-Stil
import os
os.environ["OPENAI_API_KEY"] = "sk-..."
result = agent.invoke({"messages": [...]})
print(result)

# ✅ Production-Stil
import logging
logger = logging.getLogger(__name__)

async def run_agent(query: str) -> str:
    try:
        result = await agent.ainvoke(
            {"messages": [HumanMessage(query)]},
            config={"recursion_limit": 10}
        )
        return result["messages"][-1].content
    except Exception as e:
        logger.error("Agent error", exc_info=True)
        return "Ein Fehler ist aufgetreten."
```

In [ ]:
#@markdown   <p><font size="4" color='green'>  Deployment Pipeline</font> </br></p>

diagram = '''
%%{init: {'theme':'dark'}}%%
flowchart LR
    NB["📓 Notebook\nEntwicklung"]
    PY["🐍 Python\nModule\n(agent.py)"]
    API["🌐 FastAPI\nServer\n(server.py)"]
    DC["🐳 Docker\nContainer"]
    CL["☁️ Cloud\nDeploy"]

    NB -->|Refactor| PY
    PY -->|Wrap| API
    API -->|Containerize| DC
    DC -->|Deploy| CL

    style NB fill:#37474F,color:#fff
    style PY fill:#1565C0,color:#fff
    style API fill:#2E7D32,color:#fff
    style DC fill:#4A148C,color:#fff
    style CL fill:#E65100,color:#fff
'''
mermaid(diagram, width=900)

✏️ FastAPI, Docker, uvicorn

<details>

+ **FastAPI** ist ein modernes Python‑Web‑Framework, mit dem du sehr schnell performante REST‑APIs (z.B. GET /users, POST /items) bauen kannst.
+ **Docker** ist eine Plattform, mit der du Anwendungen (z.B. deine FastAPI‑App) in isolierten „Containern“ ausführst, sodass sie auf jedem System gleich funktionieren – unabhängig von der Installation von Python, Pip‑Packages etc.
+ **Uvicorn** ist ein leichtgewichtiger, asynchroner Webserver für Python, der speziell für das ASGI‑Standard‑Interface (Asynchronous Server Gateway Interface)gedacht ist.

</details>

In [ ]:
#@markdown   <p><font size="4" color='green'>  🏗️ Production-ready Agent</font> </br></p>

import logging, os
from langchain.chat_models import init_chat_model
from langchain_core.tools import tool
from langchain_core.messages import HumanMessage
from langgraph.prebuilt import create_react_agent

# 1. Strukturiertes Logging konfigurieren
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s | %(levelname)s | %(name)s | %(message)s",
)
logger = logging.getLogger("agent.production")

# 2. LLM + Tools (API-Key kommt aus os.environ, nie hardcoded)
llm = init_chat_model("openai:gpt-4o-mini", temperature=0.0)

@tool
def suche_info(thema: str) -> str:
    '''Simuliert eine Wissenssuche zu einem Thema.'''
    daten = {
        "langchain": "LangChain ist ein Framework für LLM-Anwendungen.",
        "langgraph": "LangGraph ermöglicht zustandsbasierte Multi-Agent-Workflows.",
        "langsmith": "LangSmith bietet Tracing und Evaluation für LLM-Anwendungen.",
    }
    key = thema.lower()
    return daten.get(key, f"Keine Informationen zu '{thema}' gefunden.")

prod_agent = create_react_agent(llm, tools=[suche_info])

# 3. Production Wrapper mit Fehlerbehandlung und Logging
async def run_agent(query: str, session_id: str = "default") -> dict:
    '''Production-ready Agent-Aufruf mit Logging und Fehlerbehandlung.'''
    logger.info("Agent-Anfrage | session=%s | query=%s", session_id, query[:80])
    try:
        result = await prod_agent.ainvoke(
            {"messages": [HumanMessage(query)]},
            config={"recursion_limit": 10, "run_name": f"prod-{session_id}"}
        )
        answer = result["messages"][-1].content
        logger.info("Agent-Antwort | session=%s | chars=%d", session_id, len(answer))
        return {"status": "ok", "answer": answer, "session_id": session_id}
    except Exception:
        logger.error("Agent-Fehler | session=%s", session_id, exc_info=True)
        return {"status": "error", "answer": "Ein Fehler ist aufgetreten.", "session_id": session_id}

# Test
result = await run_agent("Was ist LangGraph?", session_id="demo-001")
mprint(f"**Antwort:** {result['answer']}\n\n**Status:** `{result['status']}` | **Session:** `{result['session_id']}`")

# 2b | Zentrale Modell-Konfiguration
---


<p><font color='black' size="5">
Warum eine zentrale Modell-Konfiguration?
</font></p>

In Notebook-Modulen wird jedes Modell direkt initialisiert — das macht die Rollenlogik
sichtbar und ist didaktisch gewollt. In einem Production-System ist das anders:

| Ansatz | Notebook | Production |
|--------|----------|------------|
| Modell-Init | direkt pro Zelle | zentrale Konfigurationsdatei |
| Modellwechsel | jede Zelle anpassen | eine Stelle ändern |
| Rollenlogik | sichtbar, lehrreich | gekapselt, wartbar |

Die Datei `model_config.py` definiert **alle sechs Rollen** an einer einzigen Stelle.
Ändert sich ein Modell (z. B. nach einem Provider-Update), reicht eine Zeile.

> Rollenübersicht: [Modell-Auswahl Guide](https://ralf-42.github.io/Agenten/frameworks/Modell_Auswahl_Guide.html) · [Provider-Modell-Mapping](https://ralf-42.github.io/Agenten/frameworks/Provider_Modell_Mapping.html)

In [ ]:
%%writefile model_config.py
"""
model_config.py — Zentrale Modell-Konfiguration für Production-Deployments

Alle sechs Rollen an einer Stelle. Modellwechsel erfordert nur eine Änderung hier.
Rollenlogik: https://ralf-42.github.io/Agenten/frameworks/Modell_Auswahl_Guide.html
"""
from langchain.chat_models import init_chat_model
from langchain_openai import OpenAIEmbeddings

# Baseline / Demo — schnell, günstig, Grundlagen und erste Tests
baseline_llm = init_chat_model("openai:gpt-4o-mini", temperature=0.0)

# Router / leichter Reasoner — einfache Routing- und Auswahlentscheidungen
router_llm   = init_chat_model("openai:o3-mini")

# Judge / starker Reasoner — Supervisor, Security, Bewertung, Compliance
judge_llm    = init_chat_model("openai:o3")

# Worker / Synthese — hochwertige Text-, RAG- und strukturierte Ausgabe
worker_llm   = init_chat_model("openai:gpt-5.1")

# Coding-Worker — Code-Generierung, Refactoring, technische Agenten-Knoten
coding_llm   = init_chat_model("openai:gpt-5.1")

# Embeddings — Vektorrepräsentationen für Retrieval und RAG (kein Chat-Modell)
embed_model  = OpenAIEmbeddings(model="text-embedding-3-small")

In [ ]:
#@markdown   <p><font size="4" color='green'>  Verwendung von model_config.py</font> </br></p>

# In jedem Production-Modul: Import statt direkter Initialisierung
# from model_config import worker_llm, judge_llm, embed_model

# Beispiel: Agent mit Rollen aus der zentralen Konfiguration
from model_config import baseline_llm, judge_llm, worker_llm, embed_model
from langchain_core.tools import tool
from langgraph.prebuilt import create_react_agent

@tool
def beispiel_tool(frage: str) -> str:
    """Beantwortet eine einfache Frage."""
    return f"Antwort auf: {frage}"

# Worker-Agent aus zentraler Konfiguration
agent = create_react_agent(
    model=worker_llm,
    tools=[beispiel_tool],
    prompt="Du bist ein hilfreicher Assistent.",
)

from genai_lib.utilities import mprint
zeilen = [
    "## ✅ model_config.py geladen",
    "",
    "| Rolle | Variable | Modell |",
    "|-------|----------|--------|",
    "| Baseline / Demo | `baseline_llm` | `gpt-4o-mini` |",
    "| Judge / starker Reasoner | `judge_llm` | `o3` |",
    "| Worker / Synthese | `worker_llm` | `gpt-5.1` |",
    "| Embeddings | `embed_model` | `text-embedding-3-small` |",
]
mprint("\n".join(zeilen))

# 3 | Docker-Container für Agenten
---

Docker kapselt den Agenten mit allen Abhängigkeiten in einen **isolierten Container**.
Damit läuft der Agent auf jedem System identisch — ob Laptop, Server oder Cloud.

**Drei Dateien für ein Production-Deployment:**

| Datei | Aufgabe |
|-------|---------|
| `Dockerfile` | Baut das Container-Image (Python, Pakete, Code) |
| `docker-compose.yml` | Orchestriert Container + Umgebungsvariablen |
| `.env` | API-Keys (nie ins Git-Repository!) |

```bash
# Typischer Workflow
docker build -t mein-agent .          # Image bauen
docker-compose up -d                  # Container starten
curl http://localhost:8080/health      # Health-Check
docker-compose logs -f                # Logs verfolgen
docker-compose down                   # Container stoppen
```

In [ ]:
#@markdown   <p><font size="4" color='green'>  Docker Architektur</font> </br></p>

diagram = '''
%%{init: {'theme':'forest'}}%%
flowchart TD
    CLI(["🖥️ Client\ncurl / requests"])
    DC["🐳 Docker Container\nport 8080:8080"]

    subgraph container ["Container"]
        UV["uvicorn\nASGI Server"]
        FA["FastAPI App\nserver.py"]
        AG["LangGraph Agent\nagent.py"]
    end

    ENV[("🔑 .env\nOPENAI_API_KEY\nLANGSMITH_API_KEY")]
    LS["☁️ LangSmith\nTracing"]

    CLI -->|HTTP POST /invoke| DC
    DC --> UV --> FA --> AG
    ENV -.->|os.environ| DC
    AG -.->|Traces| LS

    style DC  fill:#1565C0,color:#fff
    style ENV fill:#4A148C,color:#fff
    style LS  fill:#2E7D32,color:#fff
    style CLI fill:#E65100,color:#fff
'''
mermaid(diagram, width=800)

<p><font size="4" color='green'>  🐳 Dockerfile + docker-compose.yml generieren</font> </br></p>

In [ ]:
%%writefile Dockerfile
FROM python:3.11-slim

WORKDIR /app

# Abhängigkeiten installieren
COPY requirements.txt .
RUN pip install --no-cache-dir -r requirements.txt

# Code kopieren
COPY agent.py server.py ./

# Port freigeben
EXPOSE 8080

# Health-Check
HEALTHCHECK --interval=30s --timeout=5s \
    CMD curl -f http://localhost:8080/health || exit 1

# Server starten
CMD ["uvicorn", "server:app", "--host", "0.0.0.0", "--port", "8080"]

In [ ]:
%%writefile docker-compose.yml
version: '3.9'
services:
  agent:
    build: .
    ports:
      - "8080:8080"
    env_file:
      - .env
    restart: unless-stopped
    healthcheck:
      test: ["CMD", "curl", "-f", "http://localhost:8080/health"]
      interval: 30s
      timeout: 5s
      retries: 3

In [ ]:
%%writefile .env.template
# API-Keys — NICHT ins Git-Repository!
OPENAI_API_KEY=sk-...
LANGSMITH_API_KEY=lsv2_...
LANGSMITH_TRACING=true
LANGSMITH_PROJECT=prod-agent
LANGSMITH_ENDPOINT=https://eu.api.smith.langchain.com

In [ ]:
import os
zeilen = [
    "🐳 **Generierte Deployment-Dateien**", "",
    "| Datei | Größe | Zweck |",
    "|-------|-------|-------|",
]
for fname, desc in [
    ("Dockerfile",        "Container-Image Definition"),
    ("docker-compose.yml","Container-Orchestrierung"),
    (".env.template",     "API-Key Template (nicht committen!)"),
]:
    size = os.path.getsize(fname) if os.path.exists(fname) else 0
    zeilen.append(f"| `{fname}` | {size} Bytes | {desc} |")
mprint("\n".join(zeilen))

# 4 | FastAPI für API-Endpoints
---

FastAPI wickelt den LangGraph-Agenten in eine **REST API** — das ist der Standard
für Production Deployments wenn kein LangGraph Platform verwendet wird.

**Standard-Endpoints für einen Agent-Service:**

| Endpoint | Methode | Beschreibung |
|----------|---------|-------------|
| `/health` | GET | Liveness-Check (Load Balancer, Docker) |
| `/agent/invoke` | POST | Synchroner Aufruf, wartet auf vollständige Antwort |
| `/agent/stream` | POST | Streaming-Antwort (Server-Sent Events) |
| `/agent/batch` | POST | Mehrere Anfragen parallel |


In [ ]:
#@markdown   <p><font size="4" color='green'>  FastAPI Architektur</font> </br></p>

diagram = '''

sequenceDiagram
    autonumber
    participant C as 🖥️ Client
    participant F as FastAPI
    participant A as LangGraph Agent
    participant O as OpenAI API
    participant L as LangSmith

    C->>F: POST /agent/invoke
    F->>A: ainvoke(input)

    activate A
    Note over A,L: Tracing aktiv via Callbacks
    A->>O: ChatCompletion Call
    O-->>A: LLM Response
    A-->>L: Log Trace (async)
    deactivate A

    F-->>C: {"answer": "...", "status": "ok"}
'''
mermaid(diagram, width=900)

In [ ]:
#@markdown   <p><font size="4" color='green'>  🌐 FastAPI Server Code</font> </br></p>

%%writefile agent_server.py
import os, logging
from fastapi import FastAPI
from pydantic import BaseModel
from langchain.chat_models import init_chat_model
from langchain_core.tools import tool
from langchain_core.messages import HumanMessage
from langgraph.prebuilt import create_react_agent

logging.basicConfig(level=logging.INFO,
    format='%(asctime)s | %(levelname)s | %(message)s')
logger = logging.getLogger('agent.api')

app = FastAPI(title='LangGraph Agent API', version='1.0')

llm = init_chat_model('openai:gpt-4o-mini', temperature=0.0)

@tool
def suche_info(thema: str) -> str:
    '''Gibt Informationen zu LangChain-Themen zurueck.'''
    daten = {
        'langchain': 'Framework fuer LLM-Anwendungen.',
        'langgraph': 'Zustandsbasierte Multi-Agent-Workflows.',
        'langsmith': 'Tracing und Evaluation fuer LLM-Anwendungen.',
    }
    return daten.get(thema.lower(), f'Keine Info zu {thema}.')

agent = create_react_agent(llm, tools=[suche_info])

class AgentRequest(BaseModel):
    query: str
    session_id: str = 'default'

@app.get('/health')
async def health():
    return {'status': 'ok', 'service': 'langgraph-agent'}

@app.post('/agent/invoke')
async def invoke(req: AgentRequest):
    logger.info('Anfrage | session=%s | query=%s', req.session_id, req.query[:60])
    try:
        result = await agent.ainvoke({'messages': [HumanMessage(req.query)]})
        answer = result['messages'][-1].content
        return {'status': 'ok', 'answer': answer, 'session_id': req.session_id}
    except Exception as e:
        logger.error('Fehler: %s', e)
        return {'status': 'error', 'answer': str(e)}

if __name__ == '__main__':
    import uvicorn
    uvicorn.run(app, host='127.0.0.1', port=8003)

In [ ]:
#@markdown   <p><font size="4" color='green'>  🌐 FastAPI Server starten</font> </br></p>

import subprocess, requests, time, sys

log_fh = open('agent_server.log', 'w')
api_process = subprocess.Popen(
    [sys.executable, 'agent_server.py'],
    stdout=log_fh, stderr=log_fh
)
print('Starte FastAPI Server auf Port 8003 (LangChain-Import ~20s)...')

time.sleep(5)

server_ok = False
for i in range(30):
    if api_process.poll() is not None:
        break
    try:
        r = requests.get('http://127.0.0.1:8003/health', timeout=2)
        if r.status_code == 200:
            server_ok = True
            break
    except Exception:
        time.sleep(1)

if server_ok:
    zeilen = [
        '## ✅ FastAPI Agent-Server laeuft auf `http://127.0.0.1:8003`',
        '',
        '| Endpoint | Methode | Beschreibung |',
        '|----------|---------|-------------|',
        '| `/health` | GET | Liveness-Check |',
        '| `/agent/invoke` | POST | Synchroner Aufruf |',
        '| `/docs` | GET | Swagger UI |',
    ]
    mprint(chr(10).join(zeilen))
else:
    log_fh.flush()
    with open('agent_server.log') as f:
        log_tail = f.read()[-1200:]
    exit_code = api_process.poll()
    zeilen = [
        f'❌ Server-Start fehlgeschlagen (Exit-Code: {exit_code})',
        '',
        '```',
        log_tail,
        '```',
    ]
    mprint(chr(10).join(zeilen))


In [ ]:
#@markdown   <p><font size="4" color='green'>  🧪 API testen</font> </br></p>

import requests, json

BASE = "http://127.0.0.1:8003"

server_ok = True
try:
    health = requests.get(f"{BASE}/health", timeout=5).json()
    print(f"Health: {health}")
except Exception as e:
    mprint(f"❌ Server nicht erreichbar: {e}" \
           "Führe zuerst die Setup-Zelle (Cell 17) aus.")
    server_ok = False

if server_ok:
    anfragen = [
        {"query": "Was ist LangChain?",  "session_id": "test-001"},
        {"query": "Erkläre LangSmith.", "session_id": "test-002"},
    ]

    zeilen = ["## 🌐 API-Test Ergebnisse", "",
              "| Session | Query | Status | Antwort (gekürzt) |",
              "|---------|-------|--------|-------------------|"
    ]
    for req in anfragen:
        try:
            resp = requests.post(f"{BASE}/agent/invoke", json=req, timeout=30).json()
            antwort_kurz = resp.get("answer", "")[:60] + "..."
            zeilen.append(f"| `{req['session_id']}` | {req['query'][:30]} | `{resp['status']}` | {antwort_kurz} |")
        except Exception as e:
            zeilen.append(f"| `{req['session_id']}` | {req['query'][:30]} | `error` | {e} |")
    mprint(chr(10).join(zeilen))


In [ ]:
#@markdown   <p><font size="4" color='green'>  🛑 API-Server Cleanup - deaktiviert</font> </br></p>

# import os

# if "api_process" in globals():
#     api_process.terminate()
#     api_process.wait(timeout=5)
#     print(f"✅ FastAPI Server (PID {api_process.pid}) beendet")

# for f in ["agent_server.py", "agent_server.log", "Dockerfile", "docker-compose.yml", ".env.template"]:
#     if os.path.exists(f):
#         os.remove(f)
#         print(f"🗑️ {f} gelöscht")

# 5 | Monitoring und Observability
---


<p><font color='black' size="5">
Drei Ebenen der Observability
</font></p>

| Ebene | Tool | Was wird gemessen? |
|-------|------|-------------------|
| **Tracing** | LangSmith | Jeder LLM-Call, Tool-Aufruf, Node-Durchlauf |
| **Logging** | Python `logging` | Fehler, Warnings, Request-Logs |
| **Metriken** | Custom + LangSmith | Latenz, Token-Kosten, Erfolgsrate |




<p><font color='black' size="5">
LangSmith Tags und Metadata
</font></p>

Mit `run_name`, `tags` und `metadata` in der `config` werden Traces in LangSmith
kategorisierbar und durchsuchbar:

```python
config = {
    "run_name":  "prod-invoke-001",
    "tags":      ["production", "v1.2", "user-facing"],
    "metadata":  {"user_id": "u123", "feature": "chat"},
    "recursion_limit": 10,
}
result = await agent.ainvoke({"messages": [...]}, config=config)
```




<p><font color='black' size="5">
Kosten-Tracking
</font></p>

LangSmith erfasst Token-Nutzung automatisch — eigenes Tracking ergänzt
Budgetgrenzen und Alerts:

```python
KOSTEN_PRO_1K_TOKENS = {"gpt-4o-mini": 0.00015}  # USD (Input)
```

In [ ]:
#@markdown   <p><font size="4" color='green'>  Observability Stack</font> </br></p>

diagram = '''
%%{init: {'theme':'forest'}}%%
flowchart TD
    A["LangGraph Agent"]

    subgraph obs ["Observability & Monitoring"]
        LS["☁️ LangSmith\n(Tracing & Evaluation)"]
        LOG["📋 Python Logging\n(Structured JSON)"]
        MET["📊 Metriken / Dashboard\n(Kosten, Latenz, Error-Rate)"]
        AL["🔔 Alerting\n(Slack / PagerDuty)"]
    end

    A -->|OpenTelemetry / Callbacks| LS
    A -->|StdOut / File| LOG

    LS -.->|Aggregiert Daten| MET
    LOG -.->|Log-based Metrics| MET

    MET -->|Schwellwert überschritten| AL
    LOG -->|Critical Error| AL

    style A   fill:#2E7D32,color:#fff
    style LS  fill:#1565C0,color:#fff
    style LOG fill:#37474F,color:#fff
    style MET fill:#E65100,color:#fff
    style AL  fill:#B71C1C,color:#fff
'''
mermaid(diagram, width=700)

In [ ]:
#@markdown   <p><font size="4" color='green'>  📊 Production Monitoring Demo</font> </br></p>

import time, logging
from langchain_core.messages import HumanMessage

# Kosten-Tabelle (USD pro 1K Tokens, Stand 2026)
KOSTEN = {
    "gpt-4o-mini":  {"input": 0.00015, "output": 0.00060},
    "gpt-4o":       {"input": 0.00250, "output": 0.01000},
    "o3":           {"input": 0.01000, "output": 0.04000},
    "gpt-5.1":      {"input": 0.00500, "output": 0.01500},
}

protokoll = []  # Laufzeitprotokoll

async def monitored_invoke(query: str, session_id: str) -> dict:
    '''Agent-Aufruf mit vollständigem Monitoring.'''
    start = time.perf_counter()
    config = {
        "run_name":  f"prod-{session_id}",
        "tags":      ["production", "m29-demo"],
        "metadata":  {"session_id": session_id, "modul": "M33"},
        "recursion_limit": 10,
    }
    try:
        result = await prod_agent.ainvoke(
            {"messages": [HumanMessage(query)]}, config=config
        )
        latenz = round((time.perf_counter() - start) * 1000)
        # Token-Nutzung aus Metadaten extrahieren
        usage = {}
        for msg in result.get("messages", []):
            meta = getattr(msg, "usage_metadata", None)
            if meta:
                usage = meta
        inp   = usage.get("input_tokens", 0)
        out   = usage.get("output_tokens", 0)
        k     = KOSTEN.get("gpt-4o-mini", {})
        kosten_usd = inp/1000*k.get("input",0) + out/1000*k.get("output",0)
        eintrag = {
            "session":  session_id,
            "latenz_ms": latenz,
            "tokens_in": inp,
            "tokens_out": out,
            "kosten_usd": round(kosten_usd, 6),
            "status": "ok",
        }
        protokoll.append(eintrag)
        return {**eintrag, "answer": result["messages"][-1].content}
    except Exception as e:
        latenz = round((time.perf_counter() - start) * 1000)
        protokoll.append({"session": session_id, "status": "error",
                           "latenz_ms": latenz, "error": str(e)})
        return {"status": "error", "answer": str(e)}

anfragen = [
    ("Was ist LangChain?",  "s001"),
    ("Erkläre LangSmith.", "s002"),
    ("Was ist LangGraph?",  "s003"),
]
for q, sid in anfragen:
    await monitored_invoke(q, sid)

# Zusammenfassung ausgeben
gesamt_kosten = sum(e.get("kosten_usd",0) for e in protokoll)
avg_latenz    = sum(e.get("latenz_ms",0) for e in protokoll) / len(protokoll)
zeilen = [
    "## 📊 Monitoring-Zusammenfassung", "",
    "| Session | Latenz (ms) | Tokens In | Tokens Out | Kosten (USD) | Status |",
    "|---------|-------------|-----------|------------|-------------|--------|",
]
for e in protokoll:
    zeilen.append(
        f"| `{e['session']}` | {e.get('latenz_ms',0)} | "
        f"{e.get('tokens_in',0)} | {e.get('tokens_out',0)} | "
        f"{e.get('kosten_usd',0):.6f} | `{e['status']}` |"
    )
zeilen += [
    "",
    f"**Gesamt-Kosten:** `{gesamt_kosten:.6f} USD`  |  "
    f"**Ø Latenz:** `{avg_latenz:.0f} ms`  |  "
    f"**Traces in LangSmith:** Projekt `M34-Production-Deployment`"
]
mprint("\n".join(zeilen))

In [ ]:
#@markdown   <p><font size="4" color='green'>  💡 LangSmith Kosten-Vergleich: Modell-Auswahl</font> </br></p>

zeilen = [
    "## 💰 Kosten-Orientierung (Modell_Auswahl_Guide v1.2)", "",
    "| Modell | Input (USD/1K) | Output (USD/1K) | Relatives Niveau | Einsatz |",
    "|--------|--------------|----------------|-----------------|---------|",
    "| `gpt-4o-mini` | 0.000150 | 0.000600 | ⭐ | Demos, Grundlagen, Worker |",
    "| `gpt-5.1`     | 0.005000 | 0.015000 | ⭐⭐⭐ | Worker, RAG-Synthese |",
    "| `o3`          | 0.010000 | 0.040000 | ⭐⭐⭐⭐⭐ | Supervisor, Judge |",
    "",
    "> **Kurs-Budget ~5 EUR:** Bei `gpt-4o-mini` reichen ~33.000 Output-Token.",
    "> Ein `o3`-Supervisor-Aufruf kostet ~66× mehr als `gpt-4o-mini`.",
    "> → Modell_Auswahl_Guide konsequent anwenden!",
]
mprint("\n".join(zeilen))

In [ ]:
#@markdown   <p><font size="4" color='green'>  LangSmith Trace-Analyse</font> </br></p>

import time as _t; _t.sleep(2)
show_trace("M34-Production-Deployment", limit=3, show_steps=True)

# 6 | Kursrückblick: Agenten-Architektur

---




Dieses Diagramm strukturiert das Konzept eines Agenten entlang von drei Ebenen: **Eigenschaften**, **Funktionen** und **Infrastruktur**.

Auf der ersten Ebene werden grundlegende **Merkmale** beschrieben, die einen Agenten auszeichnen, etwa Autonomie, Reaktionsfähigkeit oder Zielorientierung.

Daraus leiten sich auf der zweiten Ebene die zentralen **Fähigkeiten** ab, die ein Agent zur Aufgabenerfüllung benötigt – beispielsweise Planung, Nutzung von Werkzeugen oder der Umgang mit Zustand und Feedback.

Die dritte Ebene umfasst schließlich die technischen **Rahmenbedingungen**, die einen stabilen und kontrollierten Betrieb ermöglichen, wie Kontextverwaltung, Zugriff auf externe Systeme oder Sicherheitsmechanismen.

Die Zuordnung der Module (Mx) dient dabei als Referenz auf die zugrunde liegenden Inhalte und erlaubt eine gezielte Vertiefung einzelner Aspekte.



In [7]:
#@markdown   <p><font size="4" color='green'>  Agenten-Architektur: Eigenschaften, Funktionen & Infrastruktur</font> </br></p>

diagram = """
%%{init: {'theme': 'base'}}%%
graph TD
    classDef property fill:#ececff,stroke:#9370db,stroke-width:2px,color:#333
    classDef function fill:#e1f5fe,stroke:#01579b,stroke-width:2px,color:#333
    classDef infra fill:#fff3e0,stroke:#e65100,stroke-width:2px,color:#333
    classDef groupStyle fill:#f9f9f9,stroke:#d3d3d3,stroke-dasharray: 5 5

    subgraph E1["<b>🧠 1. Eigenschaften (Was einen Agenten ausmacht)</b>"]
        direction LR
        A(["<b>Autonomie</b><br/>Trifft eigenständig Entscheidungen<br/><sub>M03, M08</sub>"])
        R(["<b>Reaktionsfähigkeit</b><br/>Reagiert auf Änderungen<br/><sub>M02, M06, M10</sub>"])
        P(["<b>Proaktivität</b><br/>Verfolgt Ziele über Zeit<br/><sub>M19–M21</sub>"])
        I(["<b>Interaktionsfähigkeit</b><br/>Interagiert mit Menschen und Systemen<br/><sub>M04, M17</sub>"])
    end

    subgraph E2["<b>🛠️ 2. Funktionen (Was ein Agent können muss)</b>"]
        direction LR
        PL["<b>Planning</b><br/>Zerlegt Ziele in Schritte<br/><sub>M09, M32</sub>"]
        ME["<b>Memory</b><br/>Hält Zustand und Verlauf<br/><sub>M16, M18</sub>"]
        TU["<b>Tool Use</b><br/>Nutzt externe Werkzeuge und APIs<br/><sub>M02, M06, M30</sub>"]
        MO["<b>Monitoring</b><br/>Beobachtet Ergebnisse und Feedback<br/><sub>M15, M24</sub>"]
        DE{{"<b>Delegation (optional)</b><br/>Orchestriert Sub-Agenten<br/><sub>M20, M32, M33</sub>"}}
    end

    subgraph E3["<b>🏗️ 3. Infrastruktur (Was den Betrieb stützt)</b>"]
        direction LR
        CM[/"<b>Context Management</b><br/>Filtert und verdichtet Kontext<br/><sub>M11–M14, M22</sub>"/]
        GR[/"<b>Guardrails</b><br/>Prüft und begrenzt Aktionen<br/><sub>M10, M23</sub>"/]
        EA[/"<b>Environment Access</b><br/>Dateien, APIs, Systeme<br/><sub>M28, M30, M34</sub>"/]
        OB[/"<b>Observability</b><br/>Logs, Traces, Debugging<br/><sub>M15, M24, M34</sub>"/]
    end

    A -.-> PL
    A -.-> TU

    P -.-> PL
    P -.-> ME
    P -.-> DE

    R -.-> MO

    I -.-> TU

    PL --> CM
    ME --> CM

    TU --> EA
    TU --> GR

    MO --> OB
    DE --> OB

    class A,R,P,I property
    class PL,ME,TU,MO,DE function
    class CM,GR,EA,OB infra
    class E1,E2,E3 groupStyle
"""

mermaid(diagram, width=1300)

# A | Aufgabe
---



<p><font color='darkblue' size="4">
📌 <b>Wichtig</b>
</font></p>

Die Aufgabestellungen unten bieten Anregungen, Sie können aber auch gerne eine andere Herausforderung angehen.

**Hinweis zur Lösungshilfe:**
> In diesem Kurs dürfen und sollen Sie generative KI auch als Unterstützung beim Lernen und Entwickeln nutzen. Wenn Sie bei einer Aufgabe festhängen, können Sie zum Beispiel Gemini in Google Colab verwenden, um Fehlermeldungen besser zu verstehen, Ideen für Teilschritte zu bekommen oder Code-Varianten zu prüfen.
> <br>**Wichtig ist nur:** Die KI dient als Lern- und Entwicklungshilfe. Der Schwerpunkt des Kurses bleibt darauf, KI-Agenten selbst zu verstehen, aufzubauen und gezielt weiterzuentwickeln.



<p><font color='black' size="5">
Production-Deployment eines eigenen Agenten
</font></p>



**Aufgabe 1 — Docker-Ready Agent:**
Erweitere `agent_server.py` um einen zweiten Endpoint `/agent/batch` der eine
Liste von Anfragen parallel verarbeitet (`asyncio.gather`). Generiere ein
passendes Dockerfile und teste mit `requests`.

**Aufgabe 2 — Kosten-Budget-Guard:**
Implementiere einen `BudgetGuard` der den `monitored_invoke`-Wrapper ergänzt:
Wenn die kumulierten Kosten ein konfigurierbares Limit (z.B. `0.01 USD`) überschreiten,
lehnt er weitere Anfragen mit einem HTTP 429 ab.

**Aufgabe 3 — Vollständiger Production Stack:**
Kombiniere Security (*Agent Security & Best Practices*: Prompt-Injection-Check), Monitoring (dieses Modul: Kosten-Tracking)
und API (FastAPI) in einem einzigen `server.py`. Die Pipeline:
`Request → Security-Check → Agent → Monitoring → Response`.

> 💡 **Tipp:** `uvicorn.run(app, host="0.0.0.0", port=8080)` für Docker;
> `host="127.0.0.1"` nur für lokale Tests.